# 02 - Baselines and backtest (synthetic data)

This notebook runs the rolling-origin backtest of the baselines. A model is only useful if `rmse_vs_random_walk` is below 1.

In [ ]:
from datetime import date

import pandas as pd

from fxhedge.config import SimulatorConfig, TrainingConfig
from fxhedge.evaluation import backtest, summarize_backtest
from fxhedge.feature_engineering import align_series
from fxhedge.sample_data import simulate_macro
from fxhedge.simulator import simulate_internal_sales

START, END = date(2020, 1, 1), date(2026, 9, 30)
macro = simulate_macro(START, END, seed=42)
sales = simulate_internal_sales(SimulatorConfig(start=START, end=END), START, END, seed=42)
aligned = align_series(macro, sales)
aligned[["date", "y_fx", "y_cash"]].tail()

In [ ]:
cfg = TrainingConfig(n_splits=5)
results = [backtest(aligned, target, cfg.horizons, cfg) for target in ("y_fx", "y_cash")]
summary = summarize_backtest(pd.concat(results, ignore_index=True))
summary.round(3)

## Reading the table

- `rmse_vs_random_walk` below 1 means the model beats the random walk at that horizon.
- The exchange rate in this notebook is a random walk by construction, so no baseline should beat it there; for sales, compare the seasonal naive baseline against the random walk in the table.
- With real data, repeat the comparison for the Prophet and LightGBM hybrid using `make evaluate` after `make train`.